# Task 2: Financial Chatbot Prototype
## BCG GenAI Consulting: Rule-Based Chatbot for Financial Queries

This notebook demonstrates a simple, rule-based chatbot that responds to predefined financial queries using data from Task 1 analysis.

**Scope**: Answers 5 common financial questions for Microsoft and Apple based on 10-K data.

## 1. Import Libraries and Load Data

In [1]:
import pandas as pd
import numpy as np
from datetime import datetime

# Load the financial data from Task 1
df = pd.read_csv('../data/financial_data_raw.csv')
df['Fiscal Year'] = pd.to_datetime(df['Fiscal Year'])

print(f"Loaded financial data for {df['Ticker'].nunique()} companies")
print(f"Data covers {df['Fiscal Year'].nunique()} fiscal years")

Loaded financial data for 2 companies
Data covers 6 fiscal years


## 2. Prepare Chatbot Data and Mappings

In [2]:
# Create a dictionary of company data for quick lookup
company_data = {}
for ticker in df['Ticker'].unique():
    company_df = df[df['Ticker'] == ticker].sort_values('Fiscal Year')
    latest = company_df.iloc[-1]
    company_data[ticker] = {
        'name': latest['Company'],
        'latest_year': latest['Fiscal Year'],
        'revenue': latest['Total Revenue'],
        'net_income': latest['Net Income'],
        'assets': latest['Total Assets'],
        'liabilities': latest['Total Liabilities'],
        'ocf': latest['Operating Cash Flow'],
        'data': company_df
    }

print("Chatbot data prepared.")
print(f"Available companies: {', '.join([v['name'] for v in company_data.values()])}")

Chatbot data prepared.
Available companies: Microsoft, Apple


## 3. Define Predefined Queries and Responses

In [3]:
# Define predefined queries with response logic
PREDEFINED_QUERIES = {
    "revenue": "What is the total revenue?",
    "net_income": "How has net income changed?",
    "profitability": "Which company is more profitable?",
    "cash_flow": "What is the operating cash flow?",
    "financial_health": "What is the financial health of the companies?"
}

print("Predefined Queries:")
print("=" * 60)
for key, query in PREDEFINED_QUERIES.items():
    print(f"  {key.upper()}: {query}")

Predefined Queries:
  REVENUE: What is the total revenue?
  NET_INCOME: How has net income changed?
  PROFITABILITY: Which company is more profitable?
  CASH_FLOW: What is the operating cash flow?
  FINANCIAL_HEALTH: What is the financial health of the companies?


## 4. Implement Chatbot Logic

In [4]:
def financial_chatbot(user_query):
    """
    Simple rule-based chatbot that responds to predefined financial queries.
    
    Parameters:
        user_query (str): User's question about financial data
    
    Returns:
        str: Chatbot response
    """
    
    query_lower = user_query.lower().strip()
    
    # Query 1: Total Revenue
    if 'revenue' in query_lower and 'total' in query_lower:
        response = "\n=== TOTAL REVENUE (Latest Fiscal Year) ===\n"
        for ticker, data in company_data.items():
            response += f"{data['name']}: ${data['revenue']/1e9:.2f}B\n"
        return response
    
    # Query 2: Net Income Change
    elif ('net income' in query_lower or 'income' in query_lower) and 'change' in query_lower:
        response = "\n=== NET INCOME COMPARISON ===\n"
        for ticker, data in company_data.items():
            company_df = data['data']
            if len(company_df) >= 2:
                latest = company_df.iloc[-1]['Net Income']
                previous = company_df.iloc[-2]['Net Income']
                change = ((latest - previous) / previous * 100) if previous != 0 else 0
                response += f"{data['name']}: ${latest/1e9:.2f}B (Change: {change:+.2f}%)\n"
            else:
                response += f"{data['name']}: ${data['net_income']/1e9:.2f}B (Only 1 year available)\n"
        return response
    
    # Query 3: Profitability Comparison
    elif 'profitable' in query_lower or ('profit' in query_lower and 'margin' in query_lower):
        response = "\n=== PROFITABILITY (Net Profit Margin) ===\n"
        margins = []
        for ticker, data in company_data.items():
            if pd.notna(data['revenue']) and pd.notna(data['net_income']):
                margin = (data['net_income'] / data['revenue']) * 100
                margins.append((data['name'], margin))
        
        margins.sort(key=lambda x: x[1], reverse=True)
        for i, (company, margin) in enumerate(margins, 1):
            response += f"{i}. {company}: {margin:.2f}%\n"
        return response
    
    # Query 4: Operating Cash Flow
    elif 'cash flow' in query_lower or 'ocf' in query_lower:
        response = "\n=== OPERATING CASH FLOW (Latest Year) ===\n"
        for ticker, data in company_data.items():
            if pd.notna(data['ocf']):
                response += f"{data['name']}: ${data['ocf']/1e9:.2f}B\n"
        return response
    
    # Query 5: Financial Health
    elif 'financial health' in query_lower or 'leverage' in query_lower or 'debt' in query_lower:
        response = "\n=== FINANCIAL HEALTH (Debt-to-Equity Ratio) ===\n"
        for ticker, data in company_data.items():
            if pd.notna(data['assets']) and pd.notna(data['liabilities']):
                equity = data['assets'] - data['liabilities']
                d_to_e = data['liabilities'] / equity if equity > 0 else np.nan
                status = "Conservative" if d_to_e < 1 else "Moderate" if d_to_e < 2 else "High"
                response += f"{data['name']}: D/E = {d_to_e:.2f} ({status} leverage)\n"
        return response
    
    # Default response for unknown queries
    else:
        return ("\nSorry, I can only answer the following predefined queries:\n" +
                "\n".join([f"  - {query}" for query in PREDEFINED_QUERIES.values()]) +
                "\n\nPlease rephrase your question to match one of these queries.")

print("Chatbot logic implemented.")

Chatbot logic implemented.


## 5. Test the Chatbot

In [5]:
# Test queries
test_queries = [
    "What is the total revenue?",
    "How has net income changed?",
    "Which company is more profitable?",
    "What is the operating cash flow?",
    "What is the financial health of the companies?"
]

print("\n" + "="*70)
print("CHATBOT TESTING")
print("="*70)

for i, query in enumerate(test_queries, 1):
    print(f"\n[Query {i}] USER: {query}")
    response = financial_chatbot(query)
    print(f"[Bot Response]:" + response)


CHATBOT TESTING

[Query 1] USER: What is the total revenue?
[Bot Response]:
=== TOTAL REVENUE (Latest Fiscal Year) ===
Microsoft: $281.72B
Apple: $416.16B


[Query 2] USER: How has net income changed?
[Bot Response]:
=== NET INCOME COMPARISON ===
Microsoft: $101.83B (Change: +15.54%)
Apple: $112.01B (Change: +19.50%)


[Query 3] USER: Which company is more profitable?
[Bot Response]:
=== PROFITABILITY (Net Profit Margin) ===
1. Microsoft: 36.15%
2. Apple: 26.92%


[Query 4] USER: What is the operating cash flow?
[Bot Response]:
=== OPERATING CASH FLOW (Latest Year) ===
Microsoft: $136.16B
Apple: $111.48B


[Query 5] USER: What is the financial health of the companies?
[Bot Response]:
=== FINANCIAL HEALTH (Debt-to-Equity Ratio) ===
Microsoft: D/E = 0.80 (Conservative leverage)
Apple: D/E = 3.87 (High leverage)



## 6. Interactive Chatbot Interface

In [6]:
def run_interactive_chatbot(max_iterations=5):
    """
    Run an interactive command-line chatbot interface.
    
    Parameters:
        max_iterations (int): Maximum number of interactions
    """
    print("\n" + "="*70)
    print("FINANCIAL CHATBOT - INTERACTIVE MODE")
    print("="*70)
    print("\nWelcome to the Financial Chatbot!")
    print("Ask questions about Microsoft and Apple's financial data.")
    print("Type 'quit' or 'exit' to end the conversation.\n")
    
    for iteration in range(max_iterations):
        user_input = input(f"\nYou: ").strip()
        
        if user_input.lower() in ['quit', 'exit', 'bye', 'goodbye']:
            print("\nBot: Thank you for using the Financial Chatbot. Goodbye!")
            break
        
        if not user_input:
            print("Bot: Please enter a question.")
            continue
        
        response = financial_chatbot(user_input)
        print(f"Bot: {response}")

# Uncomment to run interactive mode
run_interactive_chatbot()


FINANCIAL CHATBOT - INTERACTIVE MODE

Welcome to the Financial Chatbot!
Ask questions about Microsoft and Apple's financial data.
Type 'quit' or 'exit' to end the conversation.

Bot: 
Sorry, I can only answer the following predefined queries:
  - What is the total revenue?
  - How has net income changed?
  - Which company is more profitable?
  - What is the operating cash flow?
  - What is the financial health of the companies?

Please rephrase your question to match one of these queries.
Bot: 
=== TOTAL REVENUE (Latest Fiscal Year) ===
Microsoft: $281.72B
Apple: $416.16B

Bot: 
=== FINANCIAL HEALTH (Debt-to-Equity Ratio) ===
Microsoft: D/E = 0.80 (Conservative leverage)
Apple: D/E = 3.87 (High leverage)

Bot: Please enter a question.
Bot: Please enter a question.


## 7. Chatbot Documentation and Limitations

### How the Chatbot Works

**Architecture:**
- **Simple Rule-Based System**: Uses if-else statements to match user input keywords to predefined response templates
- **Data Source**: Financial data from Task 1 (10-K filings for Microsoft and Apple)
- **Response Generation**: Dynamically calculates metrics from loaded CSV data and formats responses

**Predefined Queries:**
1. **Revenue**: Displays total revenue for both companies in the latest fiscal year
2. **Net Income Change**: Shows net income comparison and year-over-year changes
3. **Profitability**: Ranks companies by net profit margin
4. **Operating Cash Flow**: Shows cash flow generation for each company
5. **Financial Health**: Displays debt-to-equity ratios and leverage assessment

### Limitations

1. **Limited Query Understanding**: Only recognizes exact keyword matches; cannot understand paraphrasing or complex natural language
2. **Predefined Scope**: Can only answer 5 predefined questions; no ability to handle new queries
3. **No Learning**: Does not learn from user interactions or improve responses over time
4. **Basic Context**: No conversation history; treats each query independently
5. **Data Constraints**: Limited to 2 companies (Microsoft, Apple); no Tesla data available
6. **No Semantic Understanding**: Cannot infer user intent or handle variations in phrasing
7. **Static Responses**: Responses are based on current data only; no forecast or scenario analysis

### Future Enhancements

To transform this into a production-grade chatbot, consider:
- Implement NLP (Natural Language Processing) for better query understanding
- Add semantic search using embeddings and vector databases
- Integrate ML models for better response ranking and personalization
- Build a Flask/FastAPI backend for scalability
- Create a web UI for better user experience
- Add real-time data integration from financial APIs
- Implement multi-turn conversation and context awareness